In [ ]:
import pandas as pd
import numpy as np
import joblib
import folium
import branca.colormap as cm

In [ ]:
df = pd.read_csv("../data/processed/featured_uhi_v2.csv")
model = joblib.load("../outputs/lst_model_xgb_v3.pkl")

FEATURES = ["NDVI", "NDBI", "Elevation", "Population"]
df["LST_baseline"] = model.predict(df[FEATURES])

priority = pd.read_csv("../outputs/reports/intervention_priority.csv")
print(f"Loaded {len(df)} points and {len(priority)} priority zones")

In [ ]:
# Aggregate points to a 10x10 grid, colour each cell by mean LST
n_bins = 10
lat_edges = np.linspace(df["Latitude"].min(), df["Latitude"].max(), n_bins + 1)
lon_edges = np.linspace(df["Longitude"].min(), df["Longitude"].max(), n_bins + 1)

df["lat_bin"] = pd.cut(df["Latitude"], bins=lat_edges, labels=False, include_lowest=True)
df["lon_bin"] = pd.cut(df["Longitude"], bins=lon_edges, labels=False, include_lowest=True)

cell_lst = df.groupby(["lat_bin", "lon_bin"])["LST_baseline"].mean().reset_index()

colormap = cm.LinearColormap(
    colors=["#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"],
    vmin=cell_lst["LST_baseline"].min(),
    vmax=cell_lst["LST_baseline"].max(),
    caption="Predicted Land Surface Temperature (C)"
)

center = [df["Latitude"].mean(), df["Longitude"].mean()]
m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

for _, c in cell_lst.iterrows():
    i, j = int(c["lat_bin"]), int(c["lon_bin"])
    bounds = [[lat_edges[i], lon_edges[j]], [lat_edges[i + 1], lon_edges[j + 1]]]
    folium.Rectangle(
        bounds=bounds,
        weight=0,
        fill=True,
        fill_color=colormap(c["LST_baseline"]),
        fill_opacity=0.6,
        popup=f"Mean LST: {c['LST_baseline']:.1f} C"
    ).add_to(m)

colormap.add_to(m)
print("Thermal grid built")

In [1]:
for rank, row in priority.iterrows():
    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        popup=(f"Priority #{rank+1}<br>"
               f"Zone {row['zone']}<br>"
               f"Baseline LST: {row['LST_baseline']:.1f} C<br>"
               f"Strategy: {row['best_strategy']}<br>"
               f"Cooling: {row['best_cooling']:.2f} C"),
        icon=folium.Icon(color="darkred", icon="fire", prefix="fa")
    ).add_to(m)

m.save("../outputs/aurangabad_heat_map.html")
print("Saved -> outputs/aurangabad_heat_map.html")
m

NameError: name 'priority' is not defined